# Phase 1 — Data Understanding & Cleaning
**Capstone 1 | Phases 1–5**

Loads the dataset, inspects class balance, audits missing values, and applies cleaning.

- **Label convention:** `0` = Human-written, `1` = AI-generated
- **Dataset:** 1,367 balanced samples across 8 content types
- **Missing values:** imputed via column means (flesch_reading_ease, gunning_fog_index, passive_voice_ratio, sentiment_score)

**Saves:** `outputs/df_cleaned.pkl`  
**Next:** `phase2_eda_visualization.ipynb`


In [ ]:
import numpy as np, random, os
RANDOM_SEED = 42
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)
try:
    import torch
    torch.manual_seed(RANDOM_SEED); torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
    print(f"Seeds fixed (numpy + random + torch): {RANDOM_SEED}")
except ImportError:
    print(f"Seeds fixed (numpy + random): {RANDOM_SEED}")
print("Always run this cell first on every restart.")


In [ ]:
import sys
!{sys.executable} -m pip install --quiet pandas numpy scikit-learn matplotlib seaborn \
    nltk textblob wordcloud textaugment tqdm
import nltk
for pkg in ['wordnet', 'stopwords', 'averaged_perceptron_tagger']:
    nltk.download(pkg, quiet=True)
print("Dependencies ready.")


In [ ]:
# neccessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from textblob import TextBlob
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from collections import Counter
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import spacy

df=pd.read_csv("/content/ai_human_content_detection_dataset.csv")
df.head()

,text_content,content_type,word_count,character_count,sentence_count,lexical_diversity,avg_sentence_length,avg_word_length,punctuation_ratio,flesch_reading_ease,gunning_fog_index,grammar_errors,passive_voice_ratio,predictability_score,burstiness,sentiment_score,label
0,Score each cause. Quality throughout beautiful...,academic_paper,288,1927,54,0.9514,5.33,5.69,0.0280,53.08,7.41,1,0.1041,105.86,0.5531,0.2034,1
1,Board its rock. Job worker break tonight coupl...,essay,253,1719,45,0.9723,5.62,5.80,0.0262,50.32,8.10,6,0.2045,100.29,0.5643,0.4854,1
2,Way debate decision produce. Dream necessary c...,academic_paper,420,2849,75,0.9071,5.60,5.79,0.0263,46.86,7.86,5,0.2308,96.88,0.4979,-0.2369,1
3,Story turn because such during open model. Tha...,creative_writing,196,1310,34,0.9592,5.76,5.69,0.0260,53.80,7.00,2,0.1912,88.79,0.6241,NaN,1
4,Place specific as simply leader fall analysis....,news_article,160,1115,28,0.9688,5.71,5.97,0.0251,44.53,8.29,0,0.1318,26.15,0.2894,NaN,1


In [ ]:
# 2. Data Understanding - Initial Data Analysis

# Class distribution
print("Class Distribution:")
print(df['label'].value_counts())
print("\n")

# Data completeness
print("Missing values:")
print(df.isnull().sum())
print("\n")

# Text characteristics
print("Text Characteristics (Descriptive Statistics):")
display(df[['word_count', 'character_count', 'sentence_count', 'lexical_diversity', 'avg_sentence_length', 'avg_word_length', 'punctuation_ratio', 'flesch_reading_ease', 'gunning_fog_index', 'grammar_errors', 'passive_voice_ratio', 'predictability_score', 'burstiness', 'sentiment_score']].describe())

# Check for duplicate rows
duplicate_rows = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_rows}")

# Check unique values of content_type
if 'content_type' in df.columns:
    print("\nUnique values in 'content_type' column:")
    print(df['content_type'].unique())

Class Distribution:
label
0    684
1    683
Name: count, dtype: int64


Missing values:
text_content             0
content_type             0
word_count               0
character_count          0
sentence_count           0
lexical_diversity        0
avg_sentence_length      0
avg_word_length          0
punctuation_ratio        0
flesch_reading_ease     79
gunning_fog_index       35
grammar_errors           0
passive_voice_ratio     31
predictability_score     0
burstiness               0
sentiment_score         54
label                    0
dtype: int64


Text Characteristics (Descriptive Statistics):


,word_count,character_count,sentence_count,lexical_diversity,avg_sentence_length,avg_word_length,punctuation_ratio,flesch_reading_ease,gunning_fog_index,grammar_errors,passive_voice_ratio,predictability_score,burstiness,sentiment_score
count,1367.000000,1367.000000,1367.000000,1367.000000,1367.000000,1367.000000,1367.000000,1288.000000,1332.000000,1367.000000,1336.000000,1367.000000,1367.000000,1313.000000
mean,140.190929,940.329188,25.610095,0.967646,5.486423,5.717783,0.027440,52.183377,7.556877,1.537674,0.150198,62.779049,0.427041,-0.007997
std,97.410218,654.335255,17.867480,0.026254,0.447202,0.279636,0.002801,10.466570,1.866676,1.912012,0.056738,28.223550,0.199249,0.588354
min,3.000000,14.000000,1.000000,0.875000,3.000000,4.000000,0.019400,-50.010000,1.200000,0.000000,0.050000,20.030000,0.101100,-0.999300
25%,61.500000,410.500000,11.000000,0.951550,5.270000,5.590000,0.026100,47.712500,6.620000,0.000000,0.099675,39.015000,0.250000,-0.525800
50%,131.000000,882.000000,24.000000,0.969200,5.480000,5.710000,0.027200,52.190000,7.515000,1.000000,0.151350,56.820000,0.408500,-0.006200
75%,193.000000,1294.500000,35.000000,0.989100,5.700000,5.830000,0.028400,57.322500,8.390000,3.000000,0.200150,86.645000,0.594300,0.502800
max,443.000000,2966.000000,83.000000,1.000000,8.000000,8.330000,0.071400,98.870000,27.870000,10.000000,0.250000,119.930000,0.799500,0.995900


Number of duplicate rows: 0

Unique values in 'content_type' column:
['academic_paper' 'essay' 'creative_writing' 'news_article' 'blog_post'
 'article' 'social_media' 'product_review']


## data cleaning

In [ ]:
# handling missing values
print("Missing values before imputation:")
numerical_features = [
    'word_count', 'character_count', 'sentence_count', 'lexical_diversity',
    'avg_sentence_length', 'avg_word_length', 'punctuation_ratio',
    'flesch_reading_ease', 'gunning_fog_index', 'grammar_errors',
    'passive_voice_ratio', 'predictability_score', 'burstiness', 'sentiment_score'
]

print(df[numerical_features].isnull().sum())

# strategy is to impute missing values with the mean or median of the column.
# in this case  using the mean for numerical features.
for feature in numerical_features:
    if df[feature].isnull().any():
        mean_val = df[feature].mean()
        df[feature] = df[feature].fillna(mean_val)


print("\nMissing values after imputation:")
print(df[numerical_features].isnull().sum())

Missing values before imputation:
word_count               0
character_count          0
sentence_count           0
lexical_diversity        0
avg_sentence_length      0
avg_word_length          0
punctuation_ratio        0
flesch_reading_ease     79
gunning_fog_index       35
grammar_errors           0
passive_voice_ratio     31
predictability_score     0
burstiness               0
sentiment_score         54
dtype: int64

Missing values after imputation:
word_count              0
character_count         0
sentence_count          0
lexical_diversity       0
avg_sentence_length     0
avg_word_length         0
punctuation_ratio       0
flesch_reading_ease     0
gunning_fog_index       0
grammar_errors          0
passive_voice_ratio     0
predictability_score    0
burstiness              0
sentiment_score         0
dtype: int64


### Summary
Initial Data Understanding and Cleaning:

* Initial exploration revealed two balanced classes (human vs. AI content).

* Missing Values: Numerical features like flesch_reading_ease, gunning_fog_index, passive_voice_ratio, and sentiment_score had missing values. These were imputed using the mean of their respective columns to ensure data completeness.

* Duplicate Rows: No duplicate rows were found in the dataset.


In [ ]:
import joblib, os
os.makedirs('outputs', exist_ok=True)
joblib.dump(df, 'outputs/df_cleaned.pkl')
print(f"Saved outputs/df_cleaned.pkl — shape: {df.shape}")
